In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import h5py
from pathlib import Path

In [ ]:
base_dir      = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
transport_dir = base_dir / "Updated Code" / "transport modified param ranges"
mat_path = transport_dir / "compare_num_trials.mat"

In [3]:
def load_compare_num_trials(mat_path):
    """
    Load compare_num_trials.mat (MATLAB v7.3 / HDF5).
    Each param set was run once with num_trials_max=10000; the num_trials=1000
    sample is just the first 1000 of those trials (nested subsample of the
    same underlying run), reconstructed here rather than duplicated in MATLAB.
    Returns:
      param_sets : list of dicts {'p0': float, 'trial_data': list of np.ndarray}
                   trial_data has 10000 entries per param set
      weights    : (num_param_sets, 3) array
    """
    param_sets = []
    with h5py.File(mat_path, 'r') as f:
        weights = f['weights'][()].T  # (num_param_sets, 3)
        atbd = f['all_trial_bkt_data_10000']  # (1, N) object refs
        for i in range(atbd.shape[1]):
            ps_grp = f[atbd[0, i]]
            p0 = float(ps_grp['p0'][()].flat[0])
            td = ps_grp['trial_data']  # (1, 10000) object refs
            trial_data = []
            for j in range(td.shape[1]):
                arr = f[td[0, j]][()]
                if arr.dtype == np.float64:
                    trial_data.append(arr.flatten())
                else:
                    trial_data.append(np.array([], dtype=np.float64))
            param_sets.append({'p0': p0, 'trial_data': trial_data})
    return param_sets, weights


def recover_theta(w):
    disc = 1*w[0] + 3*(1 - w[0])         # [1, 3]
    b    = 0.01e-3*w[1] + 0.3e-3*(1 - w[1])  # [0.01e-3, 0.3e-3]
    lr   = 0.02*w[2] + 0.5*(1 - w[2])    # [0.02, 0.5]
    return disc, lr, b


param_sets, weights = load_compare_num_trials(mat_path)
print(f"Loaded {len(param_sets)} param sets, {len(param_sets[0]['trial_data'])} trials each")

Loaded 5 param sets, 10000 trials each


In [4]:
# Quantiles and plot settings — adjust as needed after first inspection
Np      = 1000
p_vals  = [0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9]
colors  = plt.cm.plasma(np.linspace(0.1, 0.9, len(p_vals)))
N_BINS  = 60
NUM_TRIALS_SMALL = 1000
NUM_TRIALS_LARGE = 10000
print('Quantiles:', p_vals)

Quantiles: [0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9]


In [5]:
def quantile_bkt(bkt, p, Np):
    """pth-percentile breakthrough time for one trial, or None if undefined
    (fewer than p*Np particles broke through in this trial)."""
    k = int(np.floor(p * Np))
    if k < len(bkt):
        return bkt[k]
    return None


def mean_bkt(bkt):
    """Mean breakthrough time among particles that broke through in this
    trial, or None if none did (disconnected trial)."""
    if len(bkt) == 0:
        return None
    return np.mean(bkt)


def hist_curve(values, n_bins):
    """log10-space histogram (bin centers, density) of an array of times."""
    log_vals = np.log10(values)
    counts, edges = np.histogram(log_vals, bins=n_bins, density=True)
    centres = (edges[:-1] + edges[1:]) / 2
    return centres, counts

In [ ]:
pdf_path = transport_dir / '[per quantile] distrib of pth percentile bkt TO COMPARE NUM_TRIALS.pdf'

with PdfPages(pdf_path) as pdf:
    for ps_idx in range(len(param_sets)):
        ps = param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])

        trials_small = ps['trial_data'][:NUM_TRIALS_SMALL]
        trials_large = ps['trial_data'][:NUM_TRIALS_LARGE]

        fig, (ax_q, ax_m) = plt.subplots(1, 2, figsize=(14, 5))

        # --- left: pth-percentile BKT distribution, num_trials=1000 (solid) vs 10000 (dashed) ---
        for p, color in zip(p_vals, colors):
            for trials, style, n_label in (
                    (trials_small, '-',  NUM_TRIALS_SMALL),
                    (trials_large, '--', NUM_TRIALS_LARGE)):
                vals = [quantile_bkt(bkt, p, Np) for bkt in trials]
                vals = np.array([v for v in vals if v is not None])
                if len(vals) < 2:
                    continue
                centres, counts = hist_curve(vals, N_BINS)
                ax_q.plot(centres, counts, style, color=color, alpha=0.9,
                          label=f'p={p:.3f}, n_trials={n_label} (n={len(vals)})')

        ax_q.set_xlabel('breakthrough time (s)')
        ax_q.set_ylabel('probability density')
        ticks = ax_q.get_xticks()
        ax_q.set_xticks(ticks)
        ax_q.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
        ax_q.set_title('pth-percentile BKT: solid=1000 trials, dashed=10000 trials')
        ax_q.legend(fontsize=6, ncol=2)
        ax_q.grid(True, alpha=0.3)

        # --- right: mean BKT distribution, num_trials=1000 (solid) vs 10000 (dashed) ---
        for trials, style, n_label, color in (
                (trials_small, '-',  NUM_TRIALS_SMALL, 'tab:blue'),
                (trials_large, '--', NUM_TRIALS_LARGE, 'tab:red')):
            mvals = [mean_bkt(bkt) for bkt in trials]
            mvals = np.array([v for v in mvals if v is not None])
            if len(mvals) < 2:
                continue
            centres, counts = hist_curve(mvals, N_BINS)
            ax_m.plot(centres, counts, style, color=color,
                      label=f'n_trials={n_label} (n={len(mvals)})')

        ax_m.set_xlabel('breakthrough time (s)')
        ax_m.set_ylabel('probability density')
        ticks = ax_m.get_xticks()
        ax_m.set_xticks(ticks)
        ax_m.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
        ax_m.set_title('mean BKT: solid=1000 trials, dashed=10000 trials')
        ax_m.legend(fontsize=8)
        ax_m.grid(True, alpha=0.3)

        fig.suptitle(
            f'Param set {ps_idx+1}  disc={disc:.2f}, lr={lr:.2f}, '
            f'b={b:.2e}, p0={ps["p0"]:.2f}')
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}')

/var/folders/zm/fpq2g01j3r3f3yfy0phwy54c0000gn/T/ipykernel_30514/292375644.py:32: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax_q.legend(fontsize=6, ncol=2)
/var/folders/zm/fpq2g01j3r3f3yfy0phwy54c0000gn/T/ipykernel_30514/292375644.py:53: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax_m.legend(fontsize=8)
/var/folders/zm/fpq2g01j3r3f3yfy0phwy54c0000gn/T/ipykernel_30514/292375644.py:32: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax_q.legend(fontsize=6, ncol=2)
/var/folders/zm/fpq2g01j3r3f3yfy0phwy54c0000gn/T/ipykernel_30514/292375644.py:53: UserWarning: No artists with labels found to put in legend.  Note th

Saved /Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP/outputs/[compare num_trials] 1000 vs 10000 trials.pdf
